# Gold-Silver Pairs Trading — Full Analysis

Single notebook spanning all phases of `PLAN.md`. Each phase gets its own section so the evolution of the strategy is visible end to end.

## Sections
1. Data exploration
2. Cointegration analysis
3. Spread construction
4. OU modeling and half-life
5. Signal generation
6. Backtest (in-sample → walk-forward)
7. Performance analytics
8. Stress testing and regime analysis

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Data Exploration *(Phase 1)*

Pull GLD / SLV adjusted-close prices through yfinance. The cleaning pass uses the yfinance trading-day index as canonical (no synthetic business-day reindex — that would invent observations on NYSE holidays). Multi-day market closures (Hurricane Sandy, COVID-era halts) are logged as anomalies but never imputed.

In [ ]:
from src.config import load_config
from src.data_loader import load_pair_data

cfg = load_config(ROOT / "config.yaml")
df = load_pair_data(cfg, refresh=False)
print(f"rows       : {len(df):,}")
print(f"date range : {df.index.min().date()} -> {df.index.max().date()}")
print(f"NaNs       : {int(df.isna().sum().sum())}")
df.head()

In [ ]:
df[["gold","silver"]].describe().round(2)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.plot(df.index, df["gold"], color="#c9a227", label="GLD")
ax2.plot(df.index, df["silver"], color="#9aa0a6", label="SLV")
ax1.set_ylabel("GLD ($)", color="#c9a227")
ax2.set_ylabel("SLV ($)", color="#9aa0a6")
ax1.set_title("GLD & SLV adjusted close")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ret = df[["log_gold","log_silver"]].diff().dropna()
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0,0].plot(ret.index, ret["log_gold"],   lw=0.5, color="#c9a227"); axes[0,0].set_title("GLD daily log return"); axes[0,0].grid(alpha=0.3)
axes[0,1].plot(ret.index, ret["log_silver"], lw=0.5, color="#9aa0a6"); axes[0,1].set_title("SLV daily log return"); axes[0,1].grid(alpha=0.3)
axes[1,0].hist(ret["log_gold"],   bins=80, color="#c9a227", alpha=0.85); axes[1,0].set_title("GLD log-return distribution"); axes[1,0].axvline(0, color='k', lw=0.5)
axes[1,1].hist(ret["log_silver"], bins=80, color="#9aa0a6", alpha=0.85); axes[1,1].set_title("SLV log-return distribution"); axes[1,1].axvline(0, color='k', lw=0.5)
plt.tight_layout()
plt.show()

print(f"GLD/SLV daily-return correlation: {ret['log_gold'].corr(ret['log_silver']):.3f}")